# Stage 4: Machine Learning Model Implementation

## Objective
Train the five classification algorithms required by the BITS M.Tech (DSE) Machine Learning Assignment 2 using the same training dataset and the preprocessing transformations prepared in Stage 3.

### Models
1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes
5. Support Vector Machine (SVM)

### Experimental rules
- Use the same train-test split created in Stage 3.
- Use the same transformed feature set for all five models.
- Establish baseline models first; do not apply resampling or class weighting in this stage.
- Use `random_state=42` where the estimator supports it.
- Save each fitted model for reuse in Stage 5 and the Streamlit application.

**Input:** `data/processed/train_processed.csv`  
**Output:** Five fitted model files in `models/`

> Run Stage 3 successfully before running this notebook.

## 4.1 Import libraries and load the processed training data

The processed training CSV contains transformed features and a `target` column. The target encoding from Stage 3 is `no = 0` and `yes = 1`.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.utils.validation import check_is_fitted

train_path = "../data/processed/train_processed.csv"
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

train_data = pd.read_csv(train_path)

print("Training data shape:", train_data.shape)
display(train_data.head())

Training data shape: (3616, 51)


,num__age,num__balance,num__day,num__campaign,num__pdays,num__previous,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,...,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep,cat__poutcome_failure,cat__poutcome_other,cat__poutcome_success,cat__poutcome_unknown,target
0,-0.864347,-0.505865,-0.492444,0.065082,-0.411541,-0.329453,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
1,-0.673204,-0.448232,-0.248716,-0.570967,0.447116,11.136987,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0
2,0.282513,-0.499461,0.726196,-0.252942,-0.411541,-0.329453,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
3,-0.673204,2.456534,0.360604,-0.570967,-0.411541,-0.329453,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
4,0.951514,1.297118,-0.004988,1.337181,-0.411541,-0.329453,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


## 4.2 Separate training features and target

`X_train` contains the transformed input features. `y_train` contains the encoded target labels.

We do not load or use the test data to fit these models. The test set remains unseen until model evaluation in Stage 5.

In [2]:
X_train = train_data.drop(columns=["target"])
y_train = train_data["target"].astype(int)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nTraining target distribution:")
display(y_train.value_counts().sort_index().rename(index={0: "no (0)", 1: "yes (1)"}).to_frame("Count"))

X_train shape: (3616, 50)
y_train shape: (3616,)

Training target distribution:


,Count
target,
no (0),3199
yes (1),417


## 4.3 Prepare model input formats

The saved `ColumnTransformer` from Stage 3 produces a transformed feature matrix. Depending on the installed scikit-learn version and configuration, it may be sparse or dense.

Gaussian Naive Bayes requires dense input. We therefore create a dense copy for that model only. The other models can use the transformed matrix directly.

In [3]:
X_train_model = X_train.to_numpy()

# GaussianNB requires dense numeric input.
if hasattr(X_train_model, "toarray"):
    X_train_dense = X_train_model.toarray()
else:
    X_train_dense = np.asarray(X_train_model)

print("General model input type:", type(X_train_model))
print("General model input shape:", X_train_model.shape)
print("Dense model input shape:", X_train_dense.shape)

General model input type: <class 'numpy.ndarray'>
General model input shape: (3616, 50)
Dense model input shape: (3616, 50)


## 4.4 Initialize the five classification models

These are baseline configurations. Hyperparameter tuning, class-imbalance experiments, and model comparison will be handled separately after baseline evaluation.

- Logistic Regression: `max_iter=1000` allows additional iterations for convergence.
- Decision Tree: `random_state=42` makes the split decisions reproducible where randomness is involved.
- KNN: starts with the scikit-learn default configuration.
- Gaussian Naive Bayes: uses the Gaussian likelihood assumption for continuous transformed features.
- SVM: `probability=True` enables probability estimates, which are needed for ROC-AUC evaluation in Stage 5.

In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    )
}

print("Models initialized:")
for name, model in models.items():
    print(f"- {name}: {model.__class__.__name__}")

Models initialized:
- Logistic Regression: LogisticRegression
- Decision Tree: DecisionTreeClassifier
- K-Nearest Neighbors: KNeighborsClassifier
- Gaussian Naive Bayes: GaussianNB
- Support Vector Machine: SVC


## 4.5 Train the models

Fit each estimator using the same training observations. Gaussian Naive Bayes receives the dense feature matrix; the other estimators receive the transformed matrix.

No evaluation metrics are calculated here. Those will be calculated consistently for all models in Stage 5.

In [5]:
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    if name == "Gaussian Naive Bayes":
        model.fit(X_train_dense, y_train)
    else:
        model.fit(X_train_model, y_train)

    fitted_models[name] = model
    print(f"{name} training completed.\n")

print("All five models trained successfully.")

Training Logistic Regression...
Logistic Regression training completed.

Training Decision Tree...
Decision Tree training completed.

Training K-Nearest Neighbors...
K-Nearest Neighbors training completed.

Training Gaussian Naive Bayes...
Gaussian Naive Bayes training completed.

Training Support Vector Machine...


C:\Users\rohit\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine training completed.

All five models trained successfully.


## 4.6 Verify fitted models

Check that every model has been fitted and can generate predictions for a small sample of the training data. This is a basic implementation check, not a measure of generalization performance.

In [6]:
sample_size = min(5, len(y_train))

for name, model in fitted_models.items():
    check_is_fitted(model)

    if name == "Gaussian Naive Bayes":
        sample_X = X_train_dense[:sample_size]
    else:
        sample_X = X_train_model[:sample_size]

    sample_predictions = model.predict(sample_X)

    print(f"{name}")
    print("Sample predictions:", sample_predictions)
    print()

Logistic Regression
Sample predictions: [0 0 0 0 0]

Decision Tree
Sample predictions: [0 0 0 0 0]

K-Nearest Neighbors
Sample predictions: [0 0 0 0 0]

Gaussian Naive Bayes
Sample predictions: [0 1 0 0 0]

Support Vector Machine
Sample predictions: [0 0 0 0 0]



## 4.7 Save all fitted models

Save each fitted estimator as a `.joblib` file. These artifacts will be loaded in Stage 5 for evaluation and later in the Streamlit application.

The model filenames are kept stable so that downstream notebooks and `app.py` can load them consistently.

In [7]:
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "K-Nearest Neighbors": "knn.joblib",
    "Gaussian Naive Bayes": "gaussian_nb.joblib",
    "Support Vector Machine": "svm.joblib"
}

for name, model in fitted_models.items():
    file_path = os.path.join(model_dir, model_files[name])
    joblib.dump(model, file_path)
    print(f"Saved {name}: {file_path}")

Saved Logistic Regression: ../models\logistic_regression.joblib
Saved Decision Tree: ../models\decision_tree.joblib
Saved K-Nearest Neighbors: ../models\knn.joblib
Saved Gaussian Naive Bayes: ../models\gaussian_nb.joblib
Saved Support Vector Machine: ../models\svm.joblib


## 4.8 Confirm saved model artifacts

List the saved model files and load each one to verify that the artifact can be read successfully.

In [9]:
for name, filename in model_files.items():
    file_path = os.path.join(model_dir, filename)

    loaded_model = joblib.load(file_path)
    check_is_fitted(loaded_model)

    print(f"{name}: verified ({filename})")

Logistic Regression: verified (logistic_regression.joblib)
Decision Tree: verified (decision_tree.joblib)
K-Nearest Neighbors: verified (knn.joblib)
Gaussian Naive Bayes: verified (gaussian_nb.joblib)
Support Vector Machine: verified (svm.joblib)


## Stage 4: Summary

Five baseline classification models have been trained using the same processed training dataset and saved for reuse.

### Expected model artifacts

```text
models/
├── preprocessor.joblib
├── label_encoder.joblib
├── logistic_regression.joblib
├── decision_tree.joblib
├── knn.joblib
├── gaussian_nb.joblib
└── svm.joblib
```

### Completion checklist
- [ ] Processed training dataset loaded
- [ ] Training features and target separated
- [ ] Five classifiers initialized
- [ ] All five classifiers trained
- [ ] Fitted models verified
- [ ] Five model artifacts saved

### Important
Training success does not indicate that a model performs well. Do not select a winner based on training predictions. In Stage 5, all five models will be evaluated on the same held-out test set using Accuracy, AUC, Precision, Recall, F1 Score, and MCC.